# LVBench 5-arm smoke reader

**Design under test** (compression isolated on BOTH the autonomous and assisted sides):

| # | Arm | run tag | tools | isolates |
|---|-----|---------|-------|----------|
| 1 | direct (64-frame skim) | `baseline` | none | base ability |
| 2 | auto: crop | `crop` | crop (model-chosen) | native localize+detail |
| 3 | auto: crop+compress | `compress` | crop+compress (model-chosen) | +does the model's OWN compression help |
| 4 | assisted: crop | `oracle_crop` | crop @ GT region | **is perception enough given perfect localization** |
| 5 | assisted: crop+compress | `oracle_both` | compress+crop @ GT region | +does compression add to a perfect crop |
| (bonus) | assisted: compress | `oracle_compress` | compress @ GT region | coverage form alone, given localization |

**Ablations that fall out:** compression-in-auto = (3−2); compression-in-assisted = (5−4); localization = (4−2) or (oracle vs auto); tools-vs-direct = (2−1),(4−1). **n is a smoke — machinery + reader validation, NOT an accuracy measurement.**

In [1]:
import sys, os, json, glob
sys.path.insert(0, "/home/cfyang/hanklin/longvt_compression")
from fast_agent import data, config
from fast_agent import viz

TAG = "smoke5"
RUNROOT = config.RUN_ROOT
ARMS = ["baseline","crop","compress","oracle_crop","oracle_compress","oracle_both"]
LABEL = {"baseline":"1.direct","crop":"2.auto-crop","compress":"3.auto-crop+comp",
         "oracle_crop":"4.asst-crop","oracle_both":"5.asst-crop+comp","oracle_compress":"(b)asst-comp"}

def load_results(arm):
    p=os.path.join(RUNROOT,f"lvbench_{arm}_{TAG}","results.jsonl")
    if not os.path.exists(p): return {}
    out={}
    for line in open(p):
        if line.strip(): r=json.loads(line); out[str(r["question_id"])]=r
    return out
res={a:load_results(a) for a in ARMS}
traj={a:{t["question_id"]:t for t in viz.load_run(os.path.join(RUNROOT,f"lvbench_{a}_{TAG}"))} for a in ARMS}
lv={r["question_id"]:r for r in data.load_lvbench()}
QIDS=sorted(res["baseline"].keys(), key=lambda q:int(q) if q.isdigit() else q)
print("arms loaded:", {a:len(res[a]) for a in ARMS})
print("qids:", QIDS)

def evidence_region(qid):
    ev=lv[qid]["evidence"]; return (min(ev),max(ev)) if ev else None
def overlap(spans, gt):
    '''fraction of the GT region covered by the union of spans, and best single-span IoU.'''
    if not gt or not spans: return 0.0, 0.0
    gs,ge=gt; gl=max(ge-gs,1e-6)
    # union coverage of GT
    segs=sorted([(max(s,gs),min(e,ge)) for s,e in spans if min(e,ge)>max(s,gs)])
    cov=0.0; cur=None
    for s,e in segs:
        if cur is None: cur=[s,e]
        elif s<=cur[1]: cur[1]=max(cur[1],e)
        else: cov+=cur[1]-cur[0]; cur=[s,e]
    if cur: cov+=cur[1]-cur[0]
    recall=cov/gl
    best_iou=0.0
    for s,e in spans:
        inter=max(0,min(e,ge)-max(s,gs)); union=max(e,s)-min(s,e)+gl-inter
        best_iou=max(best_iou, inter/union if union>0 else 0)
    return recall, best_iou
print("helpers ready")

arms loaded: {'baseline': 6, 'crop': 6, 'compress': 6, 'oracle_crop': 6, 'oracle_compress': 6, 'oracle_both': 6}
qids: ['664', '898', '903', '1246', '1898', '2367']
helpers ready


## 1. Per-question outcomes across arms (pred / correct / tool spans)

In [2]:
def tools_str(r):
    return " ".join(f"{c['name'].replace('_video','')[:4]}({c['start']:.0f}-{c['end']:.0f})" for c in r.get("tool_calls",[])) or "-"
for q in QIDS:
    meta=lv[q]; gt=evidence_region(q)
    print("="*118)
    print(f"qid {q} | {meta['task_type']} | GT {meta['time_reference']} {list(gt)} | gold {meta['answer']}")
    for a in ARMS:
        r=res[a].get(q,{})
        ok="OK " if r.get("correct") else "   "
        print(f"  {LABEL[a]:18s} {ok}pred={str(r.get('pred')):4s} | {tools_str(r)}")

qid 664 | event understanding | GT 17:49-17:55 [1069.0, 1075.0] | gold D
  1.direct              pred=None | -
  2.auto-crop           pred=A    | crop(480-490) crop(490-500) crop(500-510) crop(510-520) crop(520-530)
  3.auto-crop+comp      pred=None | comp(0-5026) comp(0-5026) comp(0-5026) comp(0-5026) comp(0-5026)
  4.asst-crop        OK pred=D    | crop(1064-1080)
  (b)asst-comp          pred=C    | comp(1064-1080)
  5.asst-crop+comp   OK pred=D    | comp(1064-1080) crop(1064-1080)
qid 898 | entity recognition | GT 13:17-13:17 [797.0, 797.0] | gold C
  1.direct           OK pred=C    | -
  2.auto-crop        OK pred=C    | crop(50-60) crop(55-65) crop(60-70) crop(65-75) crop(70-80)
  3.auto-crop+comp   OK pred=C    | comp(0-6856) crop(102-112)
  4.asst-crop        OK pred=C    | crop(789-805)
  (b)asst-comp       OK pred=C    | comp(789-805)
  5.asst-crop+comp   OK pred=C    | comp(789-805) crop(789-805)
qid 903 | reasoning | GT 37:00-42:00 [2220.0, 2520.0] | gold C
  1.direct      

## 2. Accuracy + the ablation deltas you asked for

In [3]:
def acc(a): 
    rs=res[a]; n=len(rs); c=sum(bool(x.get('correct')) for x in rs.values()); return c,n,(100*c/n if n else 0)
print(f"{'arm':20s} {'correct/n':>10s} {'acc':>7s}")
for a in ARMS:
    c,n,p=acc(a); print(f"{LABEL[a]:20s} {f'{c}/{n}':>10s} {p:>6.1f}%")
def d(a,b):
    _,_,pa=acc(a); _,_,pb=acc(b); return pa-pb
print("\n--- ablation deltas (percentage points; n tiny, directional only) ---")
print(f"  tools help autonomous?      auto-crop - direct        = {d('crop','baseline'):+.1f}")
print(f"  compression help AUTO?      (3) - (2)                 = {d('compress','crop'):+.1f}")
print(f"  perfect localization help?  asst-crop - auto-crop     = {d('oracle_crop','crop'):+.1f}")
print(f"  perception enough (asst)?   asst-crop - direct        = {d('oracle_crop','baseline'):+.1f}")
print(f"  compression help ASSISTED?  (5) - (4)                 = {d('oracle_both','oracle_crop'):+.1f}")
print(f"  compress-only vs crop (asst)  (b) - (4)               = {d('oracle_compress','oracle_crop'):+.1f}")

arm                   correct/n     acc
1.direct                    2/6   33.3%
2.auto-crop                 3/6   50.0%
3.auto-crop+comp            2/6   33.3%
4.asst-crop                 3/6   50.0%
(b)asst-comp                2/6   33.3%
5.asst-crop+comp            4/6   66.7%

--- ablation deltas (percentage points; n tiny, directional only) ---
  tools help autonomous?      auto-crop - direct        = +16.7
  compression help AUTO?      (3) - (2)                 = -16.7
  perfect localization help?  asst-crop - auto-crop     = +0.0
  perception enough (asst)?   asst-crop - direct        = +16.7
  compression help ASSISTED?  (5) - (4)                 = +16.7
  compress-only vs crop (asst)  (b) - (4)               = -16.7


## 3. Did the AUTONOMOUS agent look where the evidence is? (the key LVBench-only metric)

For the auto arms, compare the model's *self-chosen* tool spans to the GT evidence region:
`recall` = fraction of the GT region covered by the union of the agent's spans; `IoU` = best single-span overlap.
Low recall on a wrong answer = the "never gathered" failure — now measurable with ground truth (no blind judge).

In [4]:
for a in ["crop","compress"]:
    print(f"--- {LABEL[a]} ---")
    for q in QIDS:
        r=res[a].get(q,{}); gt=evidence_region(q)
        spans=[(c["start"],c["end"]) for c in r.get("tool_calls",[])]
        rec,iou=overlap(spans,gt)
        print(f"  qid {q}: GT-recall={rec:.2f} bestIoU={iou:.2f} correct={r.get('correct')} "
              f"nspans={len(spans)}")
    print()

--- 2.auto-crop ---
  qid 664: GT-recall=0.00 bestIoU=0.00 correct=False nspans=5
  qid 898: GT-recall=0.00 bestIoU=0.00 correct=True nspans=5
  qid 903: GT-recall=0.00 bestIoU=0.00 correct=True nspans=1
  qid 1246: GT-recall=0.00 bestIoU=0.00 correct=False nspans=5
  qid 1898: GT-recall=0.00 bestIoU=0.00 correct=True nspans=1
  qid 2367: GT-recall=0.00 bestIoU=0.00 correct=False nspans=6

--- 3.auto-crop+comp ---
  qid 664: GT-recall=1.00 bestIoU=0.00 correct=False nspans=5
  qid 898: GT-recall=0.00 bestIoU=0.00 correct=True nspans=2
  qid 903: GT-recall=1.00 bestIoU=0.04 correct=False nspans=5
  qid 1246: GT-recall=1.00 bestIoU=0.00 correct=False nspans=2
  qid 1898: GT-recall=1.00 bestIoU=0.00 correct=True nspans=1
  qid 2367: GT-recall=0.00 bestIoU=0.00 correct=False nspans=2



## 4. Flips — what tools / localization fixed or broke vs direct

In [5]:
def flips(arm, base="baseline"):
    fixed=[]; broke=[]
    for q in QIDS:
        a_ok=res[arm].get(q,{}).get("correct"); b_ok=res[base].get(q,{}).get("correct")
        if a_ok and not b_ok: fixed.append(q)
        if b_ok and not a_ok: broke.append(q)
    return fixed, broke
for arm in ["crop","compress","oracle_crop","oracle_both"]:
    fx,bk=flips(arm)
    print(f"{LABEL[arm]:20s} vs direct: fixed={fx} broke={bk}")
print("\n-- the training-justifying pattern: direct wrong, auto wrong, assisted(crop) RIGHT --")
for q in QIDS:
    d_ok=res['baseline'].get(q,{}).get('correct'); a_ok=res['compress'].get(q,{}).get('correct')
    o_ok=res['oracle_crop'].get(q,{}).get('correct')
    if (not d_ok) and (not a_ok) and o_ok:
        print(f"  qid {q}: {lv[q]['task_type']} — localization was the missing piece")

2.auto-crop          vs direct: fixed=['1898'] broke=[]
3.auto-crop+comp     vs direct: fixed=['1898'] broke=['903']
4.asst-crop          vs direct: fixed=['664', '1898'] broke=['903']
5.asst-crop+comp     vs direct: fixed=['664', '1898'] broke=[]

-- the training-justifying pattern: direct wrong, auto wrong, assisted(crop) RIGHT --
  qid 664: event understanding — localization was the missing piece


## 5. Implementation checks (assert the oracle arms did what they claim)

In [6]:
import math
ok_all=True
for q in QIDS:
    gt=evidence_region(q)
    for a in ["oracle_crop","oracle_compress","oracle_both"]:
        t=traj[a].get(q)
        if not t: print(f"  MISSING traj {a} {q}"); ok_all=False; continue
        o=t.get("oracle",{})
        # forced region ~ GT (widened to >=16s floor, so allow the floor)
        reg=o.get("region"); 
        # every tool round must be oracle_forced and land on the region
        for rnd in t.get("rounds",[]):
            act=rnd.get("action") or {}
            if act.get("kind")=="tool_call":
                if not act.get("oracle_forced"): print(f"  !! {a} {q} tool not oracle_forced"); ok_all=False
                if reg and not (abs(act['start']-reg[0])<2 and abs(act['end']-reg[1])<2):
                    print(f"  !! {a} {q} {act['name']} span {act['start']:.0f}-{act['end']:.0f} != region {reg}"); ok_all=False
        # media montages exist
        for rnd in t.get("rounds",[]):
            tr=rnd.get("tool_result") or {}
            if tr.get("montage") and not os.path.exists(tr["montage"]): print(f"  !! missing montage {tr['montage']}"); ok_all=False
print("oracle arms: forced spans == GT region, oracle_forced set, montages present:", "ALL OK" if ok_all else "SEE ABOVE")
# token flow sanity for compress
for q in QIDS[:2]:
    t=traj["oracle_both"].get(q)
    for rnd in (t or {}).get("rounds",[]):
        tr=rnd.get("tool_result") or {}
        if tr.get("tool")=="compress_video":
            print(f"  qid {q} compress: base={tr['base_tokens']} kept={tr['kept_tokens']} ret={tr['retention']}")

oracle arms: forced spans == GT region, oracle_forced set, montages present: ALL OK
  qid 664 compress: base=360 kept=360 ret=1.0
  qid 898 compress: base=360 kept=360 ret=1.0


## 6. Reasoning traces (per qid, per arm)

In [7]:
def trace(t):
    for rnd in t.get("rounds",[]):
        a=rnd.get("action") or {}; k=a.get("kind"); th=(rnd.get("thinking") or "").strip().replace("\n"," ")
        if k=="tool_call":
            of=" [FORCED]" if a.get("oracle_forced") else ""
            print(f"      CALL {a['name']}({a['start']:.0f},{a['end']:.0f}){of}")
        elif k in ("answer","stop","finalizer"):
            print(f"      [{k}] {th[:300]}")
for q in QIDS:
    print("="*100); print(f"QID {q} | {lv[q]['task_type']} | {data.format_question(lv[q]).splitlines()[0]} | gold {lv[q]['answer']}")
    for a in ARMS:
        t=traj[a].get(q)
        if t: print(f"  --- {LABEL[a]} (pred {res[a][q].get('pred')} ok {res[a][q].get('correct')}) ---"); trace(t)

QID 664 | event understanding | What happens when the boy at the vending machine is about to press the button? | gold D
  --- 1.direct (pred None ok False) ---
      [answer] The provided images are a series of still frames from an animated show, likely "The Powerpuff Girls" or a similar series, given the art style and characters. The question asks what happens when a boy at a vending machine is about to press a button.  However, after reviewing all 64 provided frames, t
      [finalizer] <answer>None of the above</answer>
  --- 2.auto-crop (pred A ok False) ---
      CALL crop_video(480,490)
      CALL crop_video(490,500)
      CALL crop_video(500,510)
      CALL crop_video(510,520)
      CALL crop_video(520,530)
      [stop] <tool_call> {"name": "crop_video", "arguments": {"start_time": 530, "end_time": 540}} </tool_call>
      [finalizer] <answer>A</answer>
  --- 3.auto-crop+comp (pred None ok False) ---
      CALL compress_video(0,5026)
      CALL compress_video(0,5026)
      CALL 

## 7. Sample size + what else to watch

- **This is a smoke** (n = a handful). Deltas above are directional/implementation checks, NOT results. The real read needs the ~60-qid paired pilot.
- **Token-budget caveat:** `oracle_crop` vs `oracle_compress`/`oracle_both` are NOT token-matched on *tight* regions (a 16s region: crop ≈ few-hundred tok vs compress ≈ target budget), so read the compression ablation conditioned on the logged `kept_tokens`/`n_frames`, not as equal-budget.
- **Overlap metric interpretation:** on the pilot, the fraction of *auto-arm wrong answers with GT-recall ≈ 0* is the "never-gathered" rate — the direct measure of whether localization (not perception) is the bottleneck.
- **Pairing:** all arms share the seeded qid set; oracle arms drop the 2 malformed-`time_reference` qids, so analysis intersects on common qids.
- **Things to add for the pilot:** per-task-type breakdown (LVBench's 6 categories), decode-gap caveat (fast_agent ≈16.6pp below stock `generate`; deltas hold, absolute direct is a lower bound), and a black-frame spot-check on skim montages.